<a href="https://colab.research.google.com/github/Andysimps0n/machine-learning/blob/main/colab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
print("TensorFlow v" + tf.__version__)
print("TensorFlow Decision Forests v" + tfdf.__version__)

In [ ]:
data = pd.read_csv('./train.csv')
data.drop('Id', axis=1, inplace=True)
data.head()

In [ ]:
data.info()

In [ ]:
plt.figure(figsize=(9,8))
sns.distplot(data['SalePrice'], color='g')

In [ ]:
data_numerical = data.select_dtypes(include = ['float64', 'int64'])
data_numerical.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

In [ ]:
from sklearn.model_selection import train_test_split
train_data_pandas, valid_data_pandas = train_test_split(data, test_size=0.3, random_state=42)

In [ ]:
train_data = tfdf.keras.pd_dataframe_to_tf_dataset(train_data_pandas, label="SalePrice", task=tfdf.keras.Task.REGRESSION)
valid_data = tfdf.keras.pd_dataframe_to_tf_dataset(valid_data_pandas, label="SalePrice", task=tfdf.keras.Task.REGRESSION)

In [ ]:
model = tfdf.keras.RandomForestModel(
    hyperparameter_template="benchmark_rank1",
    task=tfdf.keras.Task.REGRESSION)

In [ ]:
model.fit(x=train_data)

In [ ]:
tfdf.model_plotter.plot_model_in_colab(model, tree_idx=0, max_depth=3)

In [ ]:
logs = model.make_inspector().training_logs()
plt.plot([log.num_trees for log in logs], [log.evaluation.rmse for log in logs])
plt.xlabel("Number of trees")
plt.ylabel("RMSE (out-of-bag)")
plt.show()

In [ ]:
inspector = model.make_inspector()
inspector.evaluation()

In [ ]:
evaluation = model.evaluate(x=valid_data,return_dict=True)

for name, value in evaluation.items():
  print(f"{name}: {value:.4f}")